In [ ]:
from analysis_helper import *

import requests
import os
import country_converter as coco

cc = coco.CountryConverter()

savefig = False

## 1. Processing Historical Data

In [ ]:
def extract_EMBER_historical_data(file_path="EMBER_historical_data.csv"):

    if os.path.exists(file_path):
        data = pd.read_csv(file_path, sep= ",")
    else:
        url = "https://storage.googleapis.com/emb-prod-bkt-publicdata/public-downloads/yearly_full_release_long_format.csv"
        try:
            response = requests.get(url)
            with open(file_path, "wb") as file:
                file.write(response.content)
            data = pd.read_csv(file_path, sep= ",")
        except requests.ConnectionError:
            logger.warning("No internet connection and file not found locally.")
            raise FileNotFoundError(
                f"File {file_path} not found and cannot download from the internet."
            )

    return data

In [ ]:
df = extract_EMBER_historical_data()

# Focus on the ASEAN region
df = df[((df.ASEAN == 1) | (df.Area == 'Timor-Leste'))]

# Get the most recent year
df = (
    df.sort_values("Year", ascending=False)
      .groupby(["Area", "Category", "Subcategory", "Variable", "Unit"], as_index=False)
      .first()   # takes the row with the latest Year because of the sorting
)

# Standardize country name
rename_country = {name: coco.convert(names=name, to='short') for name in df["Area"].unique()}
df["country"] = df["Area"].map(rename_country)

# extract electricity demand
df_load_hist = df[df.Category.isin(['Electricity demand']) & (df.Subcategory == "Demand")]
df_load_hist = df_load_hist.groupby(["country","Category","Year","Variable"])["Value"].sum().unstack(["Variable"])

# extract power generation
df_gen_hist = df[df.Category.isin(['Electricity generation']) & (df.Unit == "TWh") & (df.Subcategory == "Fuel")]
df_gen_hist = df_gen_hist.groupby(["country","Category","Year","Variable"])["Value"].sum().unstack(["Variable"])

# extract power capacity
df_cap_hist = df[df.Category.isin(['Capacity']) & (df.Subcategory == "Fuel")]
df_cap_hist = df_cap_hist.groupby(["country","Category","Year","Variable"])["Value"].sum().unstack(["Variable"])

## 2. Processing PyPSA Data

In [ ]:
def group_by_country_focus(n, c, port=""):
    """Group components by specific country and find alternative if missing."""
    from pypsa.statistics import groupers
    
    bus = f"bus{port}"
    component_buses = n.c[c].static[bus]
    buses_country = n.c.buses.static.country
    
    country = groupers._map_with_multiindex(component_buses, buses_country)
    
    if "bus1" in n.c[c].static.columns:
        missing = country.isna() | (country == "")

        component_bus1 = n.c[c].static["bus1"]
        country[missing] = groupers._map_with_multiindex(component_bus1, buses_country)[missing]

    return country.rename("country1")

pypsa.statistics.groupers.add_grouper("country1", group_by_country_focus)

tech_to_EMBER = {
    "biomass": "Bioenergy",
    'biomass EOP': "Bioenergy",
    
    "coal": "Coal",
    "lignite": "Coal",

    "CCGT": "Gas",
    "OCGT": "Gas",
    "SMR": "Gas",
    "SMR CC": "Gas",
    "gas": "Gas",

    "hydro": "Hydro",
    "ror": "Hydro",
    "PHS": "Hydro",

    "nuclear": "Nuclear",

    "B2B": "Other Fossil",
    "Fischer-Tropsch": "Other Fossil",
    "Sabatier": "Other Fossil",
    "oil": "Other Fossil",

    "geothermal": "Other Renewables",
    #"helmeth": "Other Renewables",

    "solar": "Solar",
    "solar rooftop": "Solar",

    "offwind-ac": "Wind",
    "offwind-dc": "Wind",
    "onwind": "Wind"
}

EMBER_values = list(set(tech_to_EMBER.values()))


In [ ]:
year = 2025
scenario = "baseline-existing-3H"
n = pypsa.Network(f"../results/{scenario}/postnetworks/elec_s_100_ec_lv2.0__3h_{year}_0.071_DEC_0export.nc")
rename_country = {name: coco.convert(names=name, to='short') for name in n.buses.country.unique()}

df_load = - (
    pd.DataFrame(n.statistics.energy_balance(
        groupby=["country"], components=["Load"], nice_names=False
    ))
    .reset_index()
    .assign(country=lambda df: df["country"].replace(rename_country))
    .groupby("country")[0].sum()
    .div(1e6)
)

df_gen = (
    pd.DataFrame(n.statistics.energy_balance(
        groupby=["country1", "carrier", "bus_carrier"], nice_names=False
    ))
    # Filter rows
    .reset_index()
    .assign(
        carrier=lambda df: df["carrier"].replace(tech_to_EMBER),
        country=lambda df: df["country1"].replace(rename_country)
    )
    .loc[lambda df: 
        df.bus_carrier.isin(["AC","low voltage"])
        & df.carrier.isin(EMBER_values) 
        & ~df.country.isin(["","not found"])
        ]
    .groupby(["country", "carrier"])[0].sum()
    .unstack("carrier")
    .fillna(0)
    .div(1e6)
)

efficiency = {
    "lignite": 0.33,
    "coal": 0.356,
    "OCGT": 0.405,
    "CCGT": 0.57,
}

df_cap = (
    pd.DataFrame(n.statistics.optimal_capacity(
        groupby=["country1", "carrier", "bus_carrier"], nice_names=False
    ))
    # Filter rows
    .reset_index()
    .assign(
        capacity=lambda df: df[0] * df["carrier"].map(efficiency).fillna(1),
        carrier=lambda df: df["carrier"].replace(tech_to_EMBER),
        country=lambda df: df["country1"].replace(rename_country)
    )
    .loc[lambda df: 
        df.carrier.isin(EMBER_values) 
        & ~df.component.isin(["Store"]) 
        & ~df.country.isin(["","not found"])
        & ~(df.component.isin(["Generator","Store"]) & df.carrier.isin(["Coal","Gas","Bioenergy"]))
    ]
    .groupby(["country", "carrier"])["capacity"].sum()
    .unstack("carrier")
    .fillna(0)
    .div(1e3)
)

## 3. Preparing the comparison routine

In [ ]:
fig, ax = plt.subplots(
    3, 2, figsize=(8, 10),
    gridspec_kw={'width_ratios': [2.8, 2.2]},
    sharex='col'  # share x-axis within each column
)
plt.subplots_adjust(wspace=0.05, hspace=0.05)

# Data for plots
plot_data = [
    # row 0, col 0
    (df_load, df_load_hist.groupby("country")["Demand"].sum(), "Electricity Demand [TWh/a]", 600),
    # row 1, col 0
    (df_gen.T.sum(), df_gen_hist.groupby("country").sum().T.sum(), "Generation [TWh/a]", 1000),
    # row 1, col 1
    (df_gen.sum(), df_gen_hist.groupby("country").sum().sum(), None, 1000),
    # row 2, col 0
    (df_cap.T.sum(), df_cap_hist.groupby("country").sum().T.sum(), "Capacity [GW]", 140),
    # row 2, col 1
    (df_cap.sum(), df_cap_hist.groupby("country").sum().sum(), None, 140)
]

# Plotting loop
positions = [(0, 0), (1, 0), (1, 1), (2, 0), (2, 1)]
for (data_model, data_hist, ylabel, ylim), (i, j) in zip(plot_data, positions):
    pd.DataFrame({
        "PyPSA-ASEAN": data_model,
        "Historical Data": data_hist
    }).plot.bar(ax=ax[i, j], legend=False, ylim=(0, ylim))
    if ylabel:
        ax[i, j].set_ylabel(ylabel)
    ax[i, j].grid(axis="y")
    if i != 2:  # hide xticklabels for all but bottom row
        ax[i, j].set_xticklabels([])
    if j != 0:
        ax[i, j].set_yticklabels([])

# Custom Legend (no PyPSA)

import matplotlib.patches as mpatches

legend_label = [
    f"PyPSA-ASEAN - Year 2025",
    "EMBER ASEAN Historical\nDatasets - Year 2024"
]
colors = ["tab:blue", "tab:orange"]

# Create rectangular patches for the legend
patches = [
    mpatches.Patch(color=color, label=label)
    for color, label in zip(colors, legend_label)
]

ax[0, 1].legend(
    handles=patches,
    loc="center",
    bbox_to_anchor=(0.5, 0.5),
    frameon=False,
    ncol=1,
    title="Model",
    title_fontproperties={'weight': 'bold'}
)

# Turn off the subplot where the legend is placed
ax[0, 1].axis('off')

plt.subplots_adjust(bottom=0.1)   # increase space at bottom

if savefig:
    fig.savefig(
        "validation.png",
        dpi=300,
        bbox_inches="tight",
    )